# SleepFM — embeddings NSRR pour la confirmation de l'IAH (GPU)

Fichier : `notebooks/sleepfm_nsrr_embeddings.ipynb`. Plan validé le 13 novembre 2026
(`docs/plans/Plan confirmation IAH NSRR SleepFM OSF 13-11.md`).

Embeddings SleepFM et OSF des nuits de la **liste figée** (`configs/nsrr_<cohorte>_subjects.json` :
2 056 nuits MESA, 5 793 SHHS-1), règle d'échelle `physionet_median_v2`. Les EDF sont téléchargés
par lots depuis NSRR (MD5 vérifié), lus puis supprimés ; le lot suivant se télécharge pendant le
calcul du lot en cours. Aucun label, aucun score : l'IAH se calcule ensuite en local, une seule fois
(`scripts/evaluate_nsrr_ahi.py`).

**À vérifier avant le premier run** : l'accord d'utilisation NSRR (DUA) doit autoriser le calcul sur
Kaggle (aucune donnée n'est republiée, les EDF sont supprimés après lecture et hors de la sortie).

**Une session = une partie de 400 nuits** (sortie Kaggle limitée à 20 Go et session à 12 h ;
12 Mo d'embeddings par nuit MESA). MESA : parties 0 à 5 ; SHHS-1 : parties 0 à 14.

**Sur Kaggle** :
1. `Settings > Accelerator > GPU T4 x2` et `Settings > Internet > On`.
2. `Add-ons > Secrets` : `NSRR_TOKEN` (jeton personnel sleepdata.org), attaché au notebook.
3. Essai court : `LIMIT = 2`, puis `Run All` ; vérifier le GPU, le téléchargement et le temps par nuit.
4. Partie complète : `LIMIT = None`, `PART = 0`, puis `Save Version > Save & Run All (Commit)`.
5. Télécharger depuis `Output` : `embeddings_nsrr_<cohorte>_part<N>.zip` et `nsrr_log_part<N>.txt`.
6. Recommencer avec `PART = 1`, `2`… (une version par partie).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Settings > Accelerator"
!df -h . | tail -1

## 1. Code, dépendances et poids (empreintes de la règle figée)

In [ ]:
import os, hashlib
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
REVISION = "origin/main"  # ou le SHA exact du commit, noté dans les sorties
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard {REVISION} && git log --oneline -1
%cd {WORK}/sleepfm-pipeline
!pip install -q -r requirements.txt
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
OSF_REVISION = "f2067f2d95b3bff43e4a690a471ba3d4457f12a7"
!mkdir -p third_party/osf
![ -f third_party/osf/osf_backbone.pth ] || curl -sL -o third_party/osf/osf_backbone.pth https://huggingface.co/yang-ai-lab/OSF-Base/resolve/{OSF_REVISION}/osf_backbone.pth
SLEEPFM_CKPT = "third_party/sleepfm/sleepfm/checkpoint/best.pt"
OSF_CKPT = "third_party/osf/osf_backbone.pth"
EXPECTED = {  # mêmes poids que sur PhysioNet (results/respiratory_ahi_physionet2018.json)
    SLEEPFM_CKPT: "a2fe7e5abf70f8a448c5577a085e0b7096332dcb1ec7497916e425446ac24575",
    OSF_CKPT: "c51190b1942556969af3c3d63c2e59430ddb1ea0377c50ea87df83712fc31857",
}
for path, sha in EXPECTED.items():
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    if digest != sha:
        raise RuntimeError(f"{path} : empreinte inattendue {digest}")
print("Poids SleepFM et OSF vérifiés")

## 2. Paramètres

- `COHORT` : `mesa` d'abord, puis `shhs1`.
- `PART` : numéro de la partie de 400 nuits traitée dans cette session (0, 1, 2…), ou `"rattrapage"`
  pour les seules nuits de `configs/nsrr_<cohorte>_rattrapage.json` (non obtenues lors d'une partie
  déjà passée, par exemple pendant une panne NSRR).
- `LIMIT` : `2` pour l'essai court, `None` pour la partie entière.
- `LOT` : nombre d'EDF par lot (environ 2 Go pour MESA) ; `WORKERS` : téléchargements simultanés.
- `BUDGET_H` : arrêt propre après ce nombre d'heures (lot en cours terminé, zip créé) ; les nuits
  restantes sont reprises dans une session suivante. Limite Kaggle : 12 h.

In [ ]:
COHORT = "mesa"
PART = 0
PART_SIZE = 400
LIMIT = 2  # None pour la partie entière
LOT = 10
WORKERS = 4
BUDGET_H = 11
import time
START = time.time()
SUBJECTS = f"configs/nsrr_{COHORT}_subjects.json"
SCRATCH = "/tmp/nsrr" if ON_KAGGLE else f"{WORK}/nsrr_tmp"  # hors de la sortie Kaggle
OUT = f"{SCRATCH}/embeddings/nsrr_{COHORT}"
import json
subjects = json.load(open(SUBJECTS))["subjects"]
n_parts = -(-len(subjects) // PART_SIZE)
if PART == "rattrapage":
    part = json.load(open(f"configs/nsrr_{COHORT}_rattrapage.json"))["subjects"]
    assert set(part) <= set(subjects), "rattrapage hors de la liste figée"
else:
    part = subjects[PART * PART_SIZE : (PART + 1) * PART_SIZE]
print(f"{len(subjects)} nuits dans la liste figée ; partie {PART} (0 à {n_parts - 1}) : {len(part)} nuits")
if not part:
    raise SystemExit("Partie vide : PART trop grand")

## 3. Téléchargement NSRR (code du dépôt, MD5 vérifié)

Le jeton est lu dans les secrets Kaggle et n'apparaît dans aucune sortie (`redact`).

In [ ]:
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("NSRR_TOKEN")
import shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from scripts import run_nsrr_local as nsrr

index = nsrr.folder_index(nsrr.DATASET[COHORT], nsrr.EDF_PATH[COHORT], TOKEN)
print(f"Index NSRR : {len(index)} EDF ; absents de l'index dans cette partie : {[s for s in part if f'{s}.edf' not in index]}")


def fetch_lot(lot, folder):
    """EDF du lot dans `folder` (vidé avant) ; renvoie les sujets non obtenus."""
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    return nsrr.download_lot(COHORT, lot, index, folder, TOKEN, workers=WORKERS)

## 4. Embeddings par lots

Chaque lot : `scripts/embed_nsrr.py` sur les EDF du lot (reprend les nuits déjà faites), pendant
que le lot suivant se télécharge dans l'autre dossier. Un EDF introuvable reste « manquant ».

In [ ]:
def pending():
    return [s for s in part if not (Path(OUT) / f"{s}.npz").exists() and not (Path(OUT) / f"{s}.skipped").exists()]


def command(edf_dir):
    return [
        "python", "-u", "scripts/embed_nsrr.py", "--cohort", COHORT, "--edf-dir", str(edf_dir),
        "--subjects", SUBJECTS, "--out", OUT, "--checkpoint", SLEEPFM_CKPT,
        "--respiratory-rule", "physionet_median_v2", "--osf-checkpoint", OSF_CKPT, "--batch-size", "128",
    ]


todo = pending()[:LIMIT] if LIMIT else pending()
lots = [todo[i : i + LOT] for i in range(0, len(todo), LOT)]
dirs = [Path(SCRATCH) / "edf_a", Path(SCRATCH) / "edf_b"]
unavailable = []
with open(f"{WORK}/nsrr_log_part{PART}.txt", "a") as log, ThreadPoolExecutor(1) as pool:
    future = pool.submit(fetch_lot, lots[0], dirs[0]) if lots else None
    for k, lot in enumerate(lots):
        if time.time() - START > BUDGET_H * 3600:
            print(f"Arrêt à {BUDGET_H} h : {len(lots) - k} lot(s) laissé(s) pour une prochaine session")
            break
        unavailable += future.result()
        if k + 1 < len(lots):  # lot suivant téléchargé pendant le calcul
            future = pool.submit(fetch_lot, lots[k + 1], dirs[(k + 1) % 2])
        process = subprocess.Popen(command(dirs[k % 2]), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        line = ""
        for line in process.stdout:
            log.write(line)
            if "EDF absent" not in line:
                print(line, end="")
        # code 1 normal entre deux lots seulement si l'extracteur a fini par son bilan « complet : False »
        if process.wait() != 0 and not (process.returncode == 1 and "complet : False" in line):
            raise RuntimeError(f"embed_nsrr a échoué (code {process.returncode}) : voir le journal")
        shutil.rmtree(dirs[k % 2], ignore_errors=True)
        print(f"--- lot {k + 1}/{len(lots)} : {len(part) - len(pending())}/{len(part)} nuits de la partie {PART}")
for folder in dirs:
    shutil.rmtree(folder, ignore_errors=True)
print(f"Introuvables sur NSRR : {unavailable}")

## 5. Garder le résultat

En local : `python scripts/merge_embedding_parts.py --out embeddings/nsrr_<cohorte> <zips>` (même
manifeste exigé, doublons identiques seulement), puis `embed_nsrr.py` sans EDF pour recalculer
`run_status.json`, puis évaluer une seule fois.

In [ ]:
done = [s for s in part if (Path(OUT) / f"{s}.npz").exists()]
rejected = [s for s in part if (Path(OUT) / f"{s}.skipped").exists()]
print(f"Partie {PART} : {len(done)} faites, {len(rejected)} rejetées, {len(part) - len(done) - len(rejected)} manquantes")
!python -m pip freeze > {WORK}/pip_freeze.txt
shutil.make_archive(f"{WORK}/embeddings_nsrr_{COHORT}_part{PART}", "zip", OUT)  # manifeste, NPZ, fiches, rejets
!ls -lh {WORK}/*.zip {WORK}/*.txt